# 10 Production RAG System Architecture

**Real-World Scenario**: 24/7 Customer Service Chatbot API with Streaming & Semantic Caching.

This notebook implements a **Semantic Vector Cache Simulator** ($0\text{ms}$ latency cache hits for frequent user queries), paired with a **Complete GenAI Streaming LLM Call** on cache miss.

## Step 1: Environment Setup

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from repository root .env
load_dotenv()

# Create hidden vector database directory for local index persistence
os.makedirs(".vectordb", exist_ok=True)

print("=== Repository Environment Setup ===")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))
print("Hidden DB Directory Path:", os.path.abspath(".vectordb"))


## Step 2: Semantic Cache Lookup & Execution

In [ ]:
cache_db = {
    "what is api rate limit?": "API rate limit is 100 requests per minute.",
    "how to reset password?": "Click 'Forgot Password' on login page."
}

def check_semantic_cache(query: str) -> str:
    for q, ans in cache_db.items():
        if query.lower() == q.lower():
            return f"[CACHE HIT (0ms Latency)] {ans}"
    return "[CACHE MISS -> Invoking Vector Store & LLM Pipeline]"

print("=== Semantic Vector Cache Lookup ===")
print(check_semantic_cache("what is api rate limit?"))
print(check_semantic_cache("what is database host?"))


## Step 3: Executing Complete GenAI Streaming API Call on Cache Miss

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

query = "What is the PostgreSQL primary database replica failover latency SLA?"
cache_res = check_semantic_cache(query)
print(f"Customer Query: '{query}'")
print(f"Cache Result: {cache_res}")

if "CACHE MISS" in cache_res and os.getenv("OPENAI_API_KEY"):
    prompt = ChatPromptTemplate.from_template("Answer concisely:\nQuestion: {query}\nAnswer:")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
    
    response = llm.invoke(prompt.format(query=query))
    print("\n=== Complete Streaming GenAI Response ===")
    print(response.content)
